# Weir & Goudet (2017) kinship on the GRM panel (SNPRelate::snpgdsIndivBeta), relative to the mean pair in the set

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
def hexbin_safe(ax, x, y, **kw):
    """Pair density; cells with fewer than 21 pairs are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 120, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

def read_tri(path, n):
    """Lower triangle with diagonal, row by row (GCTA .grm.bin order)."""
    tri = np.fromfile(path, dtype="float32")
    assert len(tri) == n * (n + 1) // 2, (len(tri), n)
    return tri

def tri_pairs(n, i, j):
    """Index of pair (i, j), i >= j, in that triangle."""
    i, j = np.maximum(i, j), np.minimum(i, j)
    return i * (i + 1) // 2 + j

## Kinship (SNPRelate::snpgdsIndivBeta)

In [ ]:
STAGE = "weir_goudet"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
RUN = needs_run(OUT, FORCE)
setenv(RUN=int(RUN), BED=single_target(cfg, "grm_panel"), OUT=OUT, THREADS=THREADS)

In [ ]:
%%script Rscript -
suppressMessages({ library(SNPRelate); library(data.table) })
if (Sys.getenv("RUN") != "1") { cat("exists, keeping it (FORCE = True to remake)\n"); quit(save = "no") }
env <- function(k) { v <- Sys.getenv(k); if (!nzchar(v)) stop("missing ", k); v }
bed <- env("BED"); pre <- sub("\\.bed$", "", bed)
gfile <- tempfile(tmpdir = Sys.getenv("TMPDIR", tempdir()), fileext = ".gds")
snpgdsBED2GDS(bed, paste0(pre, ".fam"), paste0(pre, ".bim"), gfile, cvt.chr = "char", verbose = FALSE)
g <- snpgdsOpen(gfile)
b <- snpgdsIndivBeta(g, inbreeding = FALSE, num.thread = as.integer(env("THREADS")), verbose = FALSE)   # diagonal = self-kinship
ids <- as.character(b$sample.id)
dir.create(dirname(env("OUT")), recursive = TRUE, showWarnings = FALSE)
writeBin(b$beta[upper.tri(b$beta, diag = TRUE)], env("OUT"), size = 4)   # symmetric: = lower triangle row by row
fwrite(data.table(id = ids), sub("\\.bin$", ".id", env("OUT")), col.names = FALSE)
jsonlite::write_json(list(n = length(ids), n_snps = length(read.gdsn(index.gdsn(g, "snp.id"))),
                          avg_val = b$avg_val, snprelate = as.character(packageVersion("SNPRelate"))),
                     sub("\\.bin$", "_meta.json", env("OUT")), auto_unbox = TRUE)
snpgdsClose(g); unlink(gfile)
cat("done\n")

In [ ]:
if RUN:
    mark_done(STAGE, OUT, tools={"SNPRelate": "snpgdsIndivBeta"})

## Against the GRM, PC-Relate and AoU's kinship

In [ ]:
meta = json.load(open(OUT.replace(".bin", "_meta.json")))
ids = pd.read_csv(OUT.replace(".bin", ".id"), header=None, dtype=str)[0]
n = len(ids)
kin = read_tri(OUT, n)
grm_id = pd.read_csv(target_items(cfg, "grm_shards")[0][0].split(".grm.bin.")[0] + ".grm.id", sep="\t", header=None, dtype=str)[1]
rep.check("Weir-Goudet and GRM have the same people in the same order", list(grm_id) == list(ids))
grm = np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")])
diag = np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1
off = np.ones(len(kin), bool); off[diag] = False
rep.note(n_snps=meta["n_snps"], mean_pair_matching=round(float(meta["avg_val"]), 5), snprelate=meta["snprelate"], self_kinship_x2_mean=round(float(2 * kin[diag].mean()), 4),
         offdiag_kinship_mean=round(float(kin[off].mean()), 5), grm_offdiag_mean=round(float(grm[off].mean()), 5))
rep.count("n", n)
rep.check("mean off-diagonal kinship ~ 0 (by construction)", abs(kin[off].mean()) < 0.005, f"{kin[off].mean():.5f}")

rel = pd.read_csv(cfg.input("relatedness"), sep="\t", dtype={"i.s": str, "j.s": str})
pos = pd.Series(np.arange(n), index=ids)
rel = rel[rel["i.s"].isin(pos.index) & rel["j.s"].isin(pos.index)]
rel = rel.assign(wg=kin[tri_pairs(n, pos[rel["i.s"]].to_numpy(), pos[rel["j.s"]].to_numpy())])
deg = lambda k: pd.cut(k, [-1, 0.0442, 0.0884, 0.177, 0.354, 1], labels=["unrelated", "3rd", "2nd", "1st", "dup/MZ"])
agree = pd.crosstab(deg(rel["kin"]).rename("aou"), deg(rel["wg"]).rename("weir_goudet"))
rep.table(agree.reset_index(), "degree_vs_aou", counts=list(agree.columns))
if len(rel):
    rep.note(aou_pairs=len(rel), corr_with_aou=round(float(np.corrcoef(rel["kin"], rel["wg"])[0, 1]), 3))
display(agree)
pcr_uri = single_target(cfg, "pcrelate")
pcr = read_tri(pcr_uri, n) if os.path.exists(pcr_uri) else None
if pcr is not None:
    rep.check("PC-Relate has the same people", list(pd.read_csv(pcr_uri.replace(".bin", ".id"), header=None, dtype=str)[0]) == list(ids))
    rep.note(corr_with_pcrelate=round(float(np.corrcoef(kin[off], pcr[off])[0, 1]), 4))

In [ ]:
others = [("GRM / 2", grm / 2)] + ([("PC-Relate", pcr)] if pcr is not None else [])
K2 = cfg.params.get("r2_k") or cfg.param("pcr_n_pcs")   # mixed sets have no round-2 gate
r2 = pd.read_csv(single_target(cfg, "round2_pca"), sep="\t", dtype={"id": str})
r2 = r2[r2["set"] == "cohort"].drop_duplicates("id").set_index("id")
X = r2.reindex(ids)[[p for p in r2 if p.startswith("PC")][:K2]].to_numpy()
i, j = np.tril_indices(n, -1)   # same order as the triangle without its diagonal
d = np.linalg.norm(X[i] - X[j], axis=1).astype("float32"); del i, j
ok = ~np.isnan(d); o = np.flatnonzero(ok)[np.argsort(d[ok], kind="stable")]   # far pairs drawn last
est = {lab: v[off] for lab, v in others + [("Weir-Goudet", kin)]}
names = list(est)
pairs = [(x, y) for a_, x in enumerate(names) for y in names[a_ + 1:]]
fig, axes = plt.subplots(2, len(pairs), figsize=(6 * len(pairs), 11), squeeze=False)
for col, (xl, yl) in enumerate(pairs):   # every off-diagonal pair; notebook only, not saved to the report
    for row, lim in enumerate([(-0.06, 0.06), (-0.1, 0.6)]):
        ax = axes[row, col]
        sc = ax.scatter(est[xl][o], est[yl][o], c=d[o], s=0.5, cmap="magma_r", alpha=0.5, lw=0, rasterized=True)
        ax.plot(lim, lim, "k:", lw=0.8)
        ax.set(xlim=lim, ylim=lim, xlabel=xl, ylabel=yl, title=f"{RUN_ID}: " + ("bulk" if row == 0 else "with relatives"))
fig.colorbar(sc, ax=axes, label=f"distance in round-2 PC space (PC1-{K2})", shrink=0.6)
plt.show()
del d, o, est

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (lo, hi) in zip(axes, [(-0.05, 0.05), (0.02, 0.6)]):
    for lab, v in others + [("Weir-Goudet", kin)]:
        v = v[off]; h, e = np.histogram(v[(v > lo) & (v < hi)], bins=120)
        ax.stairs(np.where(h >= 21, h, 0), e, label=lab)   # bins under 21 pairs not drawn
    if hi > 0.1:
        ax.set_yscale("log")
    ax.set(xlabel="kinship", title=f"{lo} to {hi} (bins with 21+ pairs)")
axes[0].legend(fontsize=8)
rep.fig(fig, "weir_goudet_hist", "off-diagonal pairs; bins with 21+ pairs"); plt.show()
rep.save()

In [ ]:
q = np.linspace(0, 1, 2001)
sets = {"pairs (off-diagonal)": off, "self (diagonal; GRM / 2 = (1 + F) / 2)": ~off}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (title, m) in zip(axes, sets.items()):   # notebook only: the curve ends are single pairs / people
    lo, hi = [], []
    for col, (lab, v) in zip(["tab:orange", "tab:blue", "tab:green"], others + [("Weir-Goudet", kin)]):
        v = v[m].astype("float64")
        x = np.quantile(v, q); med, mu = np.median(v), v.mean()
        ax.plot(x, q, color=col, label=f"{lab}: mean {mu:.4f}, median {med:.4f}")
        ax.axvline(mu, color=col, ls="--", lw=0.8); ax.axvline(med, color=col, ls=":", lw=0.8)
        lo.append(np.quantile(v, 0.001)); hi.append(np.quantile(v, 0.999))
    ax.set(xlim=(min(lo), max(hi)), xlabel="kinship", ylabel="cumulative fraction", title=f"{RUN_ID}: {title}, 0.1-99.9%")
    ax.legend(fontsize=8, title="dashed = mean, dotted = median", title_fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
## For the report: binned, bins with 21+ pairs
REL = {lab: 2 * v[off] for lab, v in others + [("Weir-Goudet", kin)]}
REL = {("GRM" if k == "GRM / 2" else k): v for k, v in REL.items()}
COLS = {"GRM": "#E69F00", "PC-Relate": "#0072B2", "Weir-Goudet": "#009E73"}
import matplotlib.patheffects as pe
HALO = [pe.Stroke(linewidth=3.5, foreground="white"), pe.Normal()]   # keeps lines visible over the density and each other
STY = {"GRM": ("o", "-"), "PC-Relate": ("s", "--"), "Weir-Goudet": ("^", ":")}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})

def binned(x, y, edges):
    """Per x bin: mean x, mean / 25% / 75% of y, pair count."""
    b = np.digitize(x, edges) - 1; ok = (b >= 0) & (b < len(edges) - 1)
    df = pd.DataFrame({"b": b[ok], "x": x[ok], "y": y[ok]}).groupby("b")
    out = pd.DataFrame({"x": df["x"].mean(), "y": df["y"].mean(), "q1": df["y"].quantile(0.25), "q3": df["y"].quantile(0.75), "n": df.size()})
    return out[out["n"] >= 21].reindex(range(len(edges) - 1))   # empty bins as NaN: lines break there

E1 = 2 * np.unique(np.round(np.r_[np.arange(-0.03, 0.06, 0.005), np.arange(0.06, 0.2, 0.02), [0.2, 0.3, 0.4, 0.6]], 4))
CMAP = "YlGnBu"   # hexes: mean PC distance of their pairs
i_, j_ = np.tril_indices(n, -1)
d = np.linalg.norm(X[i_] - X[j_], axis=1).astype("float32"); del i_, j_
ok = ~np.isnan(d)
names = list(REL); prs = [(a_, b_) for k_, a_ in enumerate(names) for b_ in names[k_ + 1:]]
fig, axes = plt.subplots(2, len(prs), figsize=(5 * len(prs), 9), squeeze=False)
for col, (xl, yl) in enumerate(prs):
  t = binned(REL[xl], REL[yl], E1)
  for row, ax in enumerate(axes[:, col]):
    lim = (-0.07, 0.12) if row == 0 else (-0.1, 0.6)
    hb = hexbin_safe(ax, REL[xl][ok], REL[yl][ok], C=d[ok], reduce_C_function=np.mean, bins=None, gridsize=150, extent=(*lim, *lim), cmap=CMAP, alpha=0.8, zorder=0)   # cells under 21 pairs blank
    ax.plot(t["x"], t["y"], "o-", ms=3.5, lw=1.3, color="k")
    ax.plot([-0.1, 0.6], [-0.1, 0.6], "k:", lw=0.8)
    ax.set(xlabel=f"{xl} relatedness", ylabel=f"{yl} relatedness", title=f"{yl} vs {xl}")
    if row == 0:
        ax.set(xlim=(-0.07, 0.12), ylim=(-0.07, 0.12))
fig.suptitle(RUN_ID)
fig.tight_layout()
fig.colorbar(hb, ax=axes, label=f"mean PC distance (PC1-{K2})", shrink=0.5)
rep.fig(fig, "estimator_comparison", "relatedness = 2 x kinship; hexes (21+ pairs): mean PC distance of their pairs; black: mean of the y estimator per x bin (21+ pairs)"); plt.show()

ED = np.quantile(d[ok], np.linspace(0, 1, 41))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, band in zip(axes, [False, True]):
    for lab, v in REL.items():
        t = binned(d[ok], v[ok], ED)
        if band:
            for q in ("q1", "q3"):
                ax.plot(t["x"], t[q], ls=STY[lab][1], lw=0.8, alpha=0.6, color=COLS[lab])
        ax.plot(t["x"], t["y"], marker=STY[lab][0], ls=STY[lab][1], ms=5, lw=1.6, color=COLS[lab], mec="white", mew=0.6, path_effects=HALO, label=lab)
    ax.axhline(0, color="k", lw=0.6)
    ax.set(xlabel=f"PC distance (round 2, PC1-{K2})", ylabel="relatedness")
axes[0].legend(frameon=False); axes[0].set_title("mean"); axes[1].set_title("mean, 25% and 75% (thin)")
fig.suptitle(RUN_ID)
fig.tight_layout()
rep.fig(fig, "relatedness_vs_pc_distance", "pairs in 40 equal-count PC-distance bins; relatedness = 2 x kinship"); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (xlim, band) in zip(axes, [((-0.07, 0.12), False), ((-0.1, 0.6), True)]):
    for lab, v in REL.items():
        t = binned(v[ok], d[ok], E1)
        if band:
            for q in ("q1", "q3"):
                ax.plot(t["x"], t[q], ls=STY[lab][1], lw=0.8, alpha=0.6, color=COLS[lab])
        ax.plot(t["x"], t["y"], marker=STY[lab][0], ls=STY[lab][1], ms=5, lw=1.6, color=COLS[lab], mec="white", mew=0.6, path_effects=HALO, label=lab)
    ax.set(xlim=xlim, xlabel="relatedness", ylabel=f"PC distance (round 2, PC1-{K2})")
axes[0].legend(frameon=False); axes[0].set_title("mean"); axes[1].set_title("mean, 25% and 75% (thin)")
fig.suptitle(RUN_ID)
fig.tight_layout()
rep.fig(fig, "pc_distance_by_relatedness", "pairs binned by relatedness (2 x kinship), bins with 21+ pairs"); plt.show()
# for the cross-group meta pages in the report PDF
HE = np.unique(np.round(np.r_[np.arange(-0.1, 0.1, 0.002), np.arange(0.1, 0.61, 0.01)], 4))
hist = pd.concat([pd.DataFrame({"estimator": k, "lo": HE[:-1], "hi": HE[1:], "n": np.histogram(v, HE)[0]}) for k, v in REL.items()])
rep.table(hist, "relatedness_hist", counts=["n"], caption="off-diagonal relatedness histogram, counts 1-20 masked")
spread = pd.DataFrame([{"estimator": k, "pairs": len(v), "mean": float(v.mean()), "sd": float(v.std()), "share_gt_0.1": float((v > 0.1).mean())} for k, v in REL.items()])
rep.table(spread, "relatedness_spread", counts=["pairs"])
del d
rep.save()